# Hamiltonian Monte Carlo: Two Knobs, Both Cliff Edges

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/zhubarb/sesen_ai_ml_tutorials/blob/main/notebooks/bayesian/hamiltonian_monte_carlo.ipynb)

A random-walk sampler proposes a step and hopes. HMC gives the sampler momentum,
rolls it across the surface of the log-density under Hamiltonian dynamics, and proposes
wherever the trajectory lands. Because the dynamics conserve energy, long moves stay
acceptable, and long moves are what a correlated target denies a random walk.

This notebook builds the leapfrog integrator and the Metropolis correction, measures HMC
against Metropolis, Gibbs and slice on the target from the previous post, then locates the
two cliff edges its parameters sit on and the two places it fails.

Companion post: [sesen.ai/blog/hamiltonian-monte-carlo-python](https://sesen.ai/blog/hamiltonian-monte-carlo-python)

Runtime: about two minutes on a CPU. Gradients are derived by hand, so there is no autodiff
library to install.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler

## 1. Effective sample size

Everything below is measured in effective samples, so the estimator comes first. This is the
initial-positive-sequence estimator Stan reports: sum the autocorrelations in adjacent pairs
and stop at the first non-positive pair.

Note the guard on a constant chain. A chain that never moved has no autocorrelation to
measure, and returning `n` there would score a frozen sampler as perfectly mixed. That is
exactly how a naive tuning sweep picks the step size that breaks everything, which happens
for real in section 7.

In [ ]:
def ess(x):
    """Geyer initial-positive-sequence effective sample size."""
    x = np.asarray(x, float)
    n = len(x)
    x = x - x.mean()
    f = np.fft.rfft(x, 2 * n)
    acf = np.fft.irfft(f * np.conjugate(f))[:n].real
    if acf[0] <= 0:
        return 0.0                    # a chain that never moved is not a well mixed one
    acf /= acf[0]
    t, total = 1, 0.0
    while t + 1 < n and acf[t] + acf[t + 1] > 0:
        total += acf[t] + acf[t + 1]
        t += 2
    return n / (1.0 + 2.0 * total)


def autocorr(x, lags):
    x = np.asarray(x, float) - np.mean(x)
    return [float(np.corrcoef(x[:-k], x[k:])[0, 1]) for k in lags]


rng = np.random.default_rng(0)
print("iid chain    ", round(ess(rng.standard_normal(2000))))
print("frozen chain ", ess(np.zeros(2000)))

## 2. The integrator

Leapfrog interleaves the position and momentum updates: half a momentum kick, L position
drifts, half a kick. Explicit Euler updates both from the same old state. Both are first-order
arithmetic and only one of them is usable.

In [ ]:
def leapfrog(z, p, grad, eps, L):
    """Symplectic. Conserves a slightly wrong energy exactly."""
    p = p + 0.5 * eps * grad(z)
    for k in range(L):
        z = z + eps * p
        if k < L - 1:
            p = p + eps * grad(z)
    p = p + 0.5 * eps * grad(z)
    return z, p


def euler(z, p, grad, eps, L):
    """Not symplectic, and the energy shows it."""
    for _ in range(L):
        g = grad(z)
        z = z + eps * p
        p = p + eps * g
    return z, p


cov = np.array([[1.0, 0.99], [0.99, 1.0]])
prec = np.linalg.inv(cov)
logp = lambda z: -0.5 * z @ prec @ z
grad = lambda z: -prec @ z
H = lambda z, p: -logp(z) + 0.5 * p @ p

z0, p0 = np.array([1.0, 1.0]), np.random.default_rng(1).standard_normal(2)
for name, integ in [("leapfrog", leapfrog), ("euler", euler)]:
    z, p = z0.copy(), p0.copy()
    trace = [H(z, p)]
    for _ in range(200):
        z, p = integ(z, p, grad, 0.15, 1)
        trace.append(H(z, p))
    drift = np.abs(np.array(trace) - trace[0])
    print(f"{name:<9} max |dH| over 200 steps: {np.nanmax(drift[np.isfinite(drift)]):.4g}")

Leapfrog's energy error oscillates within a bound. Euler's grows geometrically. Since the
acceptance probability is `min(1, exp(-dH))`, a bounded error means a bounded rejection rate at
any trajectory length, and an accumulating one means acceptance decays to zero exactly as the
trajectory gets long enough to be worth taking.

## 3. The sampler

The momentum is drawn fresh each iteration and thrown away after. The joint distribution
factorises, so a draw from the joint with the momentum discarded is a draw from the target.
The Metropolis step corrects whatever error the integrator left behind, which makes the
sampler exact for any step size that does not blow up.

`jitter` draws the trajectory length uniformly from 1..L each iteration. Section 6 shows why.

In [ ]:
def hmc(logp, grad, n, eps, L, rng, d=2, z0=None, jitter=True, div_threshold=1000.0):
    z = np.zeros(d) if z0 is None else np.asarray(z0, float).copy()
    out = np.empty((n, d))
    acc = n_grad = n_div = 0
    for i in range(n):
        steps = int(rng.integers(1, L + 1)) if jitter else L
        p = rng.standard_normal(d)
        h0 = -logp(z) + 0.5 * p @ p
        z_new, p_new = leapfrog(z, p, grad, eps, steps)
        n_grad += steps + 1
        h1 = -logp(z_new) + 0.5 * p_new @ p_new
        dh = h1 - h0
        if not np.isfinite(dh) or dh > div_threshold:
            n_div += 1                                   # divergent transition
        elif np.log(rng.random()) < -dh:
            z, acc = z_new, acc + 1
        out[i] = z
    return dict(draws=out, accept=acc / n, cost=n_grad, divergent=n_div / n)


r = hmc(logp, grad, 20000, eps=0.15, L=18, rng=np.random.default_rng(0), jitter=False)
print(f"acceptance {r['accept']:.2f}   ESS {ess(r['draws'][:, 0]):.0f} from 20000 draws")
print("sample covariance:", np.round(np.cov(r["draws"].T), 4).tolist(), " target:", cov.tolist())

## 4. Against a random walk

Both tuned, 300 draws each, on the same target.

In [ ]:
def metropolis(logp, n, step, rng, d=2):
    z = np.zeros(d)
    lp = logp(z)
    out = np.empty((n, d))
    acc, cost = 0, 1
    for i in range(n):
        prop = z + step * rng.standard_normal(d)
        lq = logp(prop)
        cost += 1
        if np.log(rng.random()) < lq - lp:
            z, lp, acc = prop, lq, acc + 1
        out[i] = z
    return dict(draws=out, accept=acc / n, cost=cost)


mh300 = metropolis(logp, 300, 1.8, np.random.default_rng(5))["draws"]
hm300 = hmc(logp, grad, 300, 0.15, 20, np.random.default_rng(5))["draws"]

g = np.linspace(-3.4, 3.4, 300)
A, B = np.meshgrid(g, g)
Z = np.exp(-0.5 * (prec[0, 0] * A**2 + 2 * prec[0, 1] * A * B + prec[1, 1] * B**2))

fig, axes = plt.subplots(1, 2, figsize=(11, 4.8), sharex=True, sharey=True)
for ax, d, name, col in [(axes[0], mh300, "Random-walk Metropolis", "#d99120"),
                         (axes[1], hm300, "Hamiltonian Monte Carlo", "#1f9e9e")]:
    ax.contour(A, B, Z, levels=np.exp(-0.5 * np.linspace(2.3, 0.4, 5) ** 2),
               colors="#8a8a8a", linewidths=0.7)
    ax.plot(d[:, 0], d[:, 1], color=col, lw=0.7, alpha=0.75)
    ax.scatter(d[:, 0], d[:, 1], s=9, color=col)
    ax.set_title(f"{name}\nESS {ess(d[:, 0]):.0f} of 300")
    ax.set_xlabel("$z_1$")
axes[0].set_ylabel("$z_2$")
plt.tight_layout()
plt.show()

## 5. The first cliff: step size

For a Gaussian, leapfrog on the eigendirection with eigenvalue `lam` is a harmonic oscillator
of frequency `1/sqrt(lam)`, and the integrator is stable only while

    eps < 2 / omega_max = 2 * sqrt(lam_min)

At rho = 0.99 the eigenvalues are 1.99 and 0.01, so the limit is exactly 0.20. Watch what
happens either side of it.

In [ ]:
lam = np.linalg.eigvalsh(cov)
print(f"eigenvalues {lam.round(4)}   condition number {lam.max()/lam.min():.0f}")
print(f"predicted stability limit: eps < 2*sqrt({lam.min():.2f}) = {2*np.sqrt(lam.min()):.3f}\n")

print(f"{'eps':>6}{'acceptance':>13}{'divergent':>12}{'ESS/gradient':>15}")
for eps in [0.05, 0.1, 0.15, 0.18, 0.2, 0.22, 0.3, 0.6]:
    r = hmc(logp, grad, 8000, eps, 20, np.random.default_rng(0))
    e = ess(r["draws"][:, 0])
    print(f"{eps:>6}{r['accept']:>13.3f}{r['divergent']:>12.3f}{e / r['cost']:>15.5f}")

An 11% increase in step size, from 0.18 to 0.20, costs a factor of 18. There is no gentle
degradation, because there is none in the underlying instability: below the threshold the
oscillator integrates stably forever, above it the energy grows geometrically.

## 6. The second cliff: trajectory length

The step size is capped by the tightest direction. The trajectory must be long enough to cross
the loosest. Along the long axis the trajectory is an oscillator with period
`2*pi*sqrt(lam_max)` in time, which at eps = 0.15 is about 59 leapfrog steps. A quarter of that
is the most decorrelating move available. A full period hands back the point it was given.

In [ ]:
eps = 0.15
print(f"quarter period {0.5*np.pi*np.sqrt(lam.max())/eps:.1f} steps, "
      f"full period {2*np.pi*np.sqrt(lam.max())/eps:.1f} steps\n")

Ls = [1, 5, 10, 15, 18, 24, 28, 32, 40, 48, 56, 60, 66, 72]
rows = []
for L in Ls:
    r = hmc(logp, grad, 10000, eps, L, np.random.default_rng(0), jitter=False)
    e = ess(r["draws"][:, 0])
    rows.append((L, e / r["cost"], autocorr(r["draws"][:, 0], [1])[0]))

print(f"{'L':>4}{'ESS/gradient':>15}{'lag-1 autocorr':>17}")
for L, g_, a in rows:
    print(f"{L:>4}{g_:>15.6f}{a:>17.3f}")

best = max(rows, key=lambda r: r[1])
worst = min(rows, key=lambda r: r[1])
print(f"\nbest L={best[0]} at {best[1]:.5f}; worst L={worst[0]} at {worst[1]:.6f}"
      f"  ->  {best[1]/worst[1]:,.0f}x")

In [ ]:
fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 4.2), sharex=True)
a1.semilogy([r[0] for r in rows], [r[1] for r in rows], color="#1f9e9e", lw=2, marker="o")
a1.axvline(0.5 * np.pi * np.sqrt(lam.max()) / eps, color="#4a6d8c", ls=":")
a1.axvline(2 * np.pi * np.sqrt(lam.max()) / eps, color="#c2453a", ls="--")
a1.set_xlabel("trajectory length L")
a1.set_ylabel("ESS per gradient evaluation")

a2.axhline(0, color="#8a8a8a", lw=0.9)
a2.plot([r[0] for r in rows], [r[2] for r in rows], color="#1f9e9e", lw=2, marker="o")
a2.axvline(0.5 * np.pi * np.sqrt(lam.max()) / eps, color="#4a6d8c", ls=":")
a2.axvline(2 * np.pi * np.sqrt(lam.max()) / eps, color="#c2453a", ls="--")
a2.set_xlabel("trajectory length L")
a2.set_ylabel("lag-1 autocorrelation")
plt.tight_layout()
plt.show()

The right panel is the mechanism directly. The lag-1 autocorrelation starts near +1, crosses
zero at the predicted quarter period, goes strongly negative near the half period where
consecutive draws land on opposite sides of the target, and returns to +1 at the full period.

The efficiency surface is therefore not unimodal, which is why you cannot tune this knob by
hill-climbing and why NUTS grows the trajectory until it doubles back instead of being told a
length. Jittering L is the cheap defence, and it costs a little peak performance:

In [ ]:
for jit in [False, True]:
    r = hmc(logp, grad, 20000, 0.15, 20, np.random.default_rng(0), jitter=jit)
    e = ess(r["draws"][:, 0])
    label = "L jittered 1-20" if jit else "fixed L = 20   "
    print(f"{label}  acceptance {r['accept']:.3f}  ESS/draw {e/20000:.3f}  "
          f"ESS/gradient {e/r['cost']:.4f}  lag-1 {autocorr(r['draws'][:, 0], [1])[0]:+.3f}")

## 7. A real posterior

Bayesian logistic regression on the Wisconsin breast-cancer data with the first four
predictors, of which mean radius and mean perimeter correlate at 0.998 because one is nearly a
linear function of the other. Collinear predictors produce a posterior ridge on their own.

The reference is the posterior mode with a Laplace covariance, computed by optimisation and so
independent of both samplers.

In [ ]:
bc = load_breast_cancer()
Xr = StandardScaler().fit_transform(bc.data[:, :4])
X = np.column_stack([np.ones(len(Xr)), Xr])
y = bc.target.astype(float)
P = X.shape[1]
print("feature correlation:\n", np.corrcoef(Xr.T).round(3))


def lr_logp(b):
    e = X @ b
    return float(y @ e - np.logaddexp(0, e).sum() - 0.5 * (b @ b) / 25.0)


def lr_grad(b):
    e = X @ b
    prob = np.where(e >= 0, 1 / (1 + np.exp(-np.abs(e))),
                    np.exp(-np.abs(e)) / (1 + np.exp(-np.abs(e))))
    return X.T @ (y - prob) - b / 25.0

In [ ]:
print(f"{'eps':>6}{'acceptance':>13}{'divergent':>12}{'min ESS/step':>15}")
for eps in [0.05, 0.1, 0.15, 0.22, 0.25, 0.3]:
    r = hmc(lr_logp, lr_grad, 8000, eps, 25, np.random.default_rng(0), d=P)
    d = r["draws"][2000:]
    print(f"{eps:>6}{r['accept']:>13.3f}{r['divergent']:>12.4f}"
          f"{min(ess(d[:, j]) for j in range(P)) / len(d):>15.4f}")

The same cliff, on a real posterior. eps = 0.22 works at 74% acceptance; eps = 0.25 rejects
every proposal and diverges on 43% of transitions.

**Note what a naive tuner would do with that row.** A chain that never moves has zero variance,
so an ESS estimator that returns `n` for a constant chain would score it as perfectly mixed.
The guard in section 1 is what stops it winning. Always read acceptance before ESS.

In [ ]:
from scipy.optimize import minimize

opt = minimize(lambda b: -lr_logp(b), np.zeros(P), jac=lambda b: -lr_grad(b), method="BFGS")
prob = 1 / (1 + np.exp(-np.clip(X @ opt.x, -500, 500)))
lap_cov = np.linalg.inv(X.T @ (X * (prob * (1 - prob))[:, None]) + np.eye(P) / 25.0)

rh = hmc(lr_logp, lr_grad, 8000, 0.22, 25, np.random.default_rng(0), d=P)
rm = metropolis(lr_logp, 8000, 0.02, np.random.default_rng(0), d=P)
dh, dm = rh["draws"][2000:], rm["draws"][2000:]

names = ["intercept"] + [str(f) for f in bc.feature_names[:4]]
print(f"{'coefficient':<18}{'Laplace':>10}{'HMC':>10}{'MH':>10}{'Laplace sd':>13}{'MH sd':>9}")
for j, nm in enumerate(names):
    print(f"{nm:<18}{opt.x[j]:>10.3f}{dh[:, j].mean():>10.3f}{dm[:, j].mean():>10.3f}"
          f"{np.sqrt(lap_cov[j, j]):>13.3f}{dm[:, j].std():>9.3f}")

print(f"\nESS on mean radius: HMC {ess(dh[:, 1]):.0f}, MH {ess(dm[:, 1]):.0f}, of 6000 draws")

Metropolis returns the wrong sign on a coefficient whose true value is near +12, and reports a
standard deviation several times too small, so its interval excludes the right answer
comfortably. It is not slow here in a way a trace plot makes obvious. It is confidently wrong,
and its own uncertainty estimate is the thing most wrong about it.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.plot(dm[:, 1], color="#d99120", lw=0.7, label=f"MH  (ESS {ess(dm[:, 1]):.0f})")
ax.plot(dh[:, 1], color="#1f9e9e", lw=0.7, label=f"HMC (ESS {ess(dh[:, 1]):.0f})")
ax.axhline(opt.x[1], color="#2b2b2b", ls="--", lw=1.2)
ax.set_xlabel("draw (after burn-in)")
ax.set_ylabel("coefficient on mean radius")
ax.legend(frameon=False)
plt.tight_layout()
plt.show()

## 8. Neal's funnel

The shape every hierarchical model takes in its centred parameterisation. Let `v ~ N(0, 3^2)`
and `x_i | v ~ N(0, exp(v))`. The width of the `x` distribution varies by a factor of `exp(6)`
across the plausible range of `v`, so no single step size fits the whole geometry.

In [ ]:
D = 9


def funnel_logp(z):
    v, x = z[0], z[1:]
    return -0.5 * (v / 3.0) ** 2 - 0.5 * (D * v + np.exp(-v) * (x @ x))


def funnel_grad(z):
    v, x = z[0], z[1:]
    gv = -v / 9.0 - 0.5 * D + 0.5 * np.exp(-v) * (x @ x)
    return np.concatenate(([gv], -np.exp(-v) * x))


print(f"{'eps':>6}{'divergent':>12}{'recovered sd of v':>20}{'deepest v':>12}   (true sd 3.00)")
for eps in [0.6, 0.3, 0.15, 0.05]:
    r = hmc(funnel_logp, funnel_grad, 8000, eps, 20, np.random.default_rng(2), d=D + 1)
    v = r["draws"][:, 0]
    print(f"{eps:>6}{r['divergent']:>12.4%}{v.std():>20.3f}{v.min():>12.2f}")

Two things from that table. The divergences do their job: at eps = 0.6 they fire on 7% of
transitions and they cluster in the neck, which is where the answer is wrong.

And the harder lesson: **zero divergences is not a clean bill of health**. At eps = 0.15 nothing
diverges and the recovered standard deviation is still well short of 3.00, because the sampler
never reaches the neck to have trouble there. A diagnostic that fires tells you something is
wrong. Silence tells you nothing either way.

The real fix is a different parameterisation, not a smaller step. Sample `x_tilde ~ N(0, 1)` and
set `x = x_tilde * exp(v/2)`, and the funnel becomes a product of independent standard normals
with no neck in it at all. Exercise 3 asks you to verify that.

## Exercises

1. **The stability limit, swept.** Build the bivariate Gaussian for rho in {0.5, 0.9, 0.99, 0.999},
   predict `2*sqrt(lam_min)` for each, then find the empirical cliff by sweeping eps. How closely
   does the prediction hold as the condition number grows?

2. **A mass matrix.** Draw the momentum from `N(0, M)` with `M` set to the inverse of the target
   covariance, and adjust the kinetic energy accordingly. This whitens the geometry. Confirm that
   the condition number the sampler sees drops to 1 and that both cliffs disappear.

3. **Non-centred funnel.** Implement the reparameterised funnel and check that HMC recovers a
   standard deviation of 3.00 for `v` at a step size where the centred version could not.

4. **The U-turn criterion.** NUTS stops doubling the trajectory when the position at one end
   starts moving back towards the other, that is when `(z_plus - z_minus) . p_minus < 0` or the
   same with `p_plus`. Implement that test and check that it selects a length near the quarter
   period without being told what the period is.

5. **Where MH wins.** Find a target on which random-walk Metropolis beats HMC per unit of work.
   Section 5 of the companion post says where to look.

## Where this comes from

- Duane, Kennedy, Pendleton and Roweth (1987), "Hybrid Monte Carlo"
- Neal (2011), "MCMC using Hamiltonian dynamics"
- Hoffman and Gelman (2014), "The No-U-Turn Sampler"
- Betancourt (2017), "A Conceptual Introduction to Hamiltonian Monte Carlo"
- Neal (2003), "Slice Sampling", for the funnel
- Beskos, Pillai, Roberts, Sanz-Serna and Stuart (2013), for the 0.65 acceptance target